# vfbd3 — restyle video with diffusion

Two engines:

- **vace** — the Wan VACE video model, run over the clip in overlapping windows. Stable, coherent motion.
- **frame** — per-frame SDXL img2img with motion-compensated feedback of the previous output (the original *controlnetvideo* technique). Trails, drift and all.

Runtime → Change runtime type → **T4 GPU** (or better). Output frames are kept in `<output>.work/`, so if the session disconnects, re-running the same cell continues where it stopped — put the output on Drive to survive a full runtime reset.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "vfbd3[upscale] @ git+https://github.com/un1tz3r0/vfbd3"
# Colab's preinstalled torchao is too old for peft, which then refuses to load any LoRA; vfbd3 doesn't use it
!pip uninstall -q -y torchao

In [ ]:
#@title Storage and input video
use_drive = True  #@param {type:"boolean"}
project_dir = "vfbd3"  #@param {type:"string"}
input_video = ""  #@param {type:"string"}
#@markdown Leave `input_video` empty to upload a file; otherwise give a path (relative to the project folder, or absolute).
import os
if use_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    base = f"/content/drive/MyDrive/{project_dir}"
else:
    base = f"/content/{project_dir}"
os.makedirs(base, exist_ok=True)
os.chdir(base)
if not input_video:
    from google.colab import files
    input_video = next(iter(files.upload()))
input_video = os.path.abspath(input_video)
print("input:", input_video, "\nworking in:", base)

## Engine 1: vace (video model)

Generates at about 480p / 16 fps and upscales afterwards. On a T4 expect several minutes per 81-frame window (about 5 seconds of video).

- `anchor = first` passes the first output frame to every window as a reference; try `none` if the first frame's composition bleeds into later ones.
- `warp_frames > 0` is experimental: it pre-fills that many frames after each overlap by flow-warping the last finished frame.

In [ ]:
#@title Run vace
output_video = "out_vace.mp4"  #@param {type:"string"}
prompt = "a colorful murakami mural, superflat, smiling flowers"  #@param {type:"string"}
control = "depth"  #@param ["depth", "canny", "gray", "none"]
control_scale = 1.0  #@param {type:"number"}
fast = "causvid"  #@param ["none", "causvid", "selfforcing"]
steps = 0  #@param {type:"integer"}
guidance = 0  #@param {type:"number"}
chunk_frames = 81  #@param [49, 65, 81] {type:"raw"}
overlap = 9  #@param [5, 9, 13] {type:"raw"}
anchor = "first"  #@param ["first", "none"]
warp_frames = 0  #@param {type:"integer"}
upscale_to = 720  #@param [0, 720, 1080, 1440] {type:"raw"}
start = 0  #@param {type:"number"}
duration = 0  #@param {type:"number"}
seed = 0  #@param {type:"integer"}
restart = False  #@param {type:"boolean"}
#@markdown `fast` loads a step-distillation LoRA; `steps = 0` and `guidance = 0` use the preset's values (30 and 5.0 without one). `duration = 0` processes the whole clip. Tick `restart` to throw away a previous partial run of this output.
from vfbd3 import vace, common
cfg = vace.VaceConfig(prompt=prompt, control=control, control_scale=control_scale, fast=None if fast == "none" else fast,
    steps=steps or None, guidance=guidance or None,
    chunk_frames=chunk_frames, overlap=overlap, anchor=anchor, warp_frames=warp_frames,
    upscale=upscale_to or None, start=start or None, duration=duration or None, seed=seed, restart=restart)
vace.run(input_video, output_video, cfg)
common.free_memory()

## Engine 2: frame (feedback loop)

Generates at 720p directly. `denoise` is how much of each frame is re-imagined; `feedthrough` mixes the raw input back into the loop (0 = pure feedback). Any SDXL checkpoint works as `model`.

In [ ]:
#@title Run frame
output_video = "out_frame.mp4"  #@param {type:"string"}
prompt = "murakami mural"  #@param {type:"string"}
negative_prompt = ""  #@param {type:"string"}
model = ""  #@param {type:"string"}
control = "depth"  #@param ["depth", "canny"]
control_scale = 1.0  #@param {type:"number"}
short_side = 720  #@param {type:"integer"}
fps = 0  #@param {type:"number"}
fast = "lcm"  #@param ["none", "lcm", "lightning", "dmd2", "turbo"]
tiny_vae = True  #@param {type:"boolean"}
steps = 0  #@param {type:"integer"}
guidance = 0  #@param {type:"number"}
denoise = 0.5  #@param {type:"slider", min:0.05, max:1.0, step:0.05}
feedthrough = 0.0  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
motion_estimator = "dis"  #@param ["dis", "farneback", "raft", "raft_small"]
occlusion = True  #@param {type:"boolean"}
occlusion_denoise = 1.0  #@param {type:"slider", min:0.05, max:1.0, step:0.05}
noise = "warp"  #@param ["warp", "fixed", "random"]
seed = 0  #@param {type:"integer"}
ip_adapter = 0.0  #@param {type:"number"}
color_fix = 0.0  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
feedback_filter = ""  #@param {type:"string"}
output_filter = ""  #@param {type:"string"}
upscale_to = 0  #@param [0, 1080, 1440, 2160] {type:"raw"}
start = 0  #@param {type:"number"}
duration = 0  #@param {type:"number"}
restart = False  #@param {type:"boolean"}
#@markdown Leave `model` empty for SDXL base (or SDXL-Turbo with `fast = turbo`). `fast` picks a few-step setup; `steps = 0` and `guidance = 0` use its values (25 and 7.5 without one), and a frame runs `steps * denoise` steps. `occlusion` re-generates newly revealed areas at `occlusion_denoise`. `noise = warp` carries one noise field along the motion. `fps = 0` keeps the input's frame rate (lower it to save time). Filter chains use the controlnetvideo syntax, e.g. `zoom rate=0.02; hueshift deg=2`; run `!vfbd3 filters` for the list.
from vfbd3 import frame, common
cfg = frame.FrameConfig(prompt=prompt, negative_prompt=negative_prompt, model=model or None, control=control,
    control_scale=control_scale, short_side=short_side, fps=fps or None, fast=None if fast == "none" else fast, tiny_vae=tiny_vae,
    steps=steps or None, guidance=guidance or None, occlusion_denoise=occlusion_denoise, noise=noise, seed=seed,
    denoise=denoise, feedthrough=feedthrough, motion_estimator=motion_estimator, occlusion=occlusion,
    ip_adapter=ip_adapter, color_fix=color_fix, feedback_filter=feedback_filter or None,
    output_filter=output_filter or None, upscale=upscale_to or None, start=start or None,
    duration=duration or None, restart=restart)
frame.run(input_video, output_video, cfg)
common.free_memory()

In [ ]:
#@title Preview the last output
from IPython.display import Video
Video(output_video, embed=True, width=720)